# Day Six

## inventory System

In [ ]:
# self updating
# first step is to load the inventory
# print rsults cant be used in two differents functions but return results can be used again in a different functions
# def save_inventory(INVENTORY_FILE,index=False) index=false means dont generate index

In [ ]:
import pandas as pd
import os
import datetime as datetime

INVENTORY_FILE = "inventory.csv"
def load_inventory():
    """Load inventory from CSV or create empty DataFrame"""
    if os.path.exists(INVENTORY_FILE):
        df = pd.read_csv(INVENTORY_FILE)
        # Ensure required columns exist
        for col in ['prod_name', 'qty', 'prices']:
            if col not in df.columns:
                df[col] = []
    else:
        df = pd.DataFrame(columns=['prod_name', 'qty', 'prices'])
    # Always recalculate derived columns
    if not df.empty:
        df['Amount'] = df['qty'] * df['prices']
        df['Discounted'] = df['Amount'] * 0.5
    return df
def save_inventory(df):
    """Save current inventory to CSV"""
    df.to_csv(INVENTORY_FILE, index=False)
    print("Inventory saved successfully.")
def display_inventory(df):
    """Pretty print inventory"""
    if df.empty:
        print("\nInventory is empty.")
        return
    print("\n" + "=" * 70)
    print(df.to_string(index=False))
    print("=" * 70)
    print(f"Total Stock Value (full price): {df['Amount'].sum():.2f}")
    print(f"Total Stock Value (50% discount): {df['Discounted'].sum():.2f}")
def add_product(df):
    """Add a new product or increase quantity of existing one"""
    name = input("Enter product name: ").strip()
    if not name:
        print("Product name cannot be empty.")
        return df
    # Check if product already exists
    existing = df[df['prod_name'].str.lower() == name.lower()]
    if not existing.empty:
        idx = existing.index[0]
        print(f"Product '{df.at[idx, 'prod_name']}' already exists.")
        try:
            add_qty = int(input("Enter additional quantity to add: "))
            df.at[idx, 'qty'] += add_qty
            print(f"Quantity updated. New qty: {df.at[idx, 'qty']}")
        except ValueError:
            print("Invalid quantity.")
    else:
        try:
            qty = int(input(f"Enter quantity of {name}: "))
            price = float(input(f"Enter price of {name}: "))
            new_row = pd.DataFrame({
                'prod_name': [name],
                'qty': [qty],
                'prices': [price]
            })
            df = pd.concat([df, new_row], ignore_index=True)
            print(f"Added '{name}' to inventory.")
        except ValueError:
            print("Invalid quantity or price.")
    # Recalculate
    df['Amount'] = df['qty'] * df['prices']
    df['Discounted'] = df['Amount'] * 0.5
    return df
def update_product(df):
    """Update quantity or price of an existing product"""
    if df.empty:
        print("Inventory is empty.")
        return df
    display_inventory(df)
    name = input("Enter product name to update: ").strip()
    match = df[df['prod_name'].str.lower() == name.lower()]
    if match.empty:
        print("Product not found.")
        return df
    idx = match.index[0]
    print(f"\nCurrent fi Qty: {df.at[idx, 'qty']}, Price: {df.at[idx, 'prices']}")
    choice = input("Update (q)uantity, (p)rice, or (b)oth? ").lower()
    try:
        if choice in ('q', 'b'):
            new_qty = int(input("New quantity: "))
            df.at[idx, 'qty'] = new_qty
        if choice in ('p', 'b'):
            new_price = float(input("New price: "))
            df.at[idx, 'prices'] = new_price
    except ValueError:
        print("Invalid input.")
        return df
    df['Amount'] = df['qty'] * df['prices']
    df['Discounted'] = df['Amount'] * 0.5
    print("Product updated.")
    return df
def remove_product(df):
    """Remove a product from inventory"""
    if df.empty:
        print("Inventory is empty.")
        return df
    display_inventory(df)
    name = input("Enter product name to remove: ").strip()
    match = df[df['prod_name'].str.lower() == name.lower()]
    if match.empty:
        print("Product not found.")
        return df
    df = df.drop(match.index).reset_index(drop=True)
    print(f"Removed '{name}'.")
    return df
def low_stock_alert(df, threshold=5):
    """Show products with low stock"""
    if df.empty:
        print("Inventory is empty.")
        return
    low = df[df['qty'] <= threshold]
    if low.empty:
        print(f"No products with quantity £ {threshold}.")
    else:
        print(f"\nn Low stock (qty £ {threshold}):")
        print(low[['prod_name', 'qty']].to_string(index=False))
def generate_receipt(df):
    """Generate a simple receipt from current inventory"""
    if df.empty:
        print("Inventory is empty. Cannot generate receipt.")
        return
    display_inventory(df)
    print("\nEnter products for the receipt (leave blank to finish):")
    receipt_items = []
    while True:
        name = input("Product name: ").strip()
        if not name:
            break
        match = df[df['prod_name'].str.lower() == name.lower()]
        if match.empty:
            print("Product not found in inventory.")
            continue
        idx = match.index[0]
        try:
            qty = int(input(f"Quantity of {df.at[idx, 'prod_name']}: "))
            if qty > df.at[idx, 'qty']:
                print(f"Only {df.at[idx, 'qty']} available.")
                continue
            # Deduct from inventory
            df.at[idx, 'qty'] -= qty
            price = df.at[idx, 'prices']
            amount = qty * price
            discounted = amount * 0.5
            receipt_items.append({
                'Product': df.at[idx, 'prod_name'],
                'Qty': qty,
                'Unit Price': price,
                'Amount': amount,
                'Discounted (50%)': discounted
            })
        except ValueError:
            print("Invalid quantity.")
    if not receipt_items:
        print("No items added to receipt.")
        return df
    receipt_df = pd.DataFrame(receipt_items)
    print("\n" + "=" * 60)
    print(f"RECEIPT - {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("=" * 60)
    print(receipt_df.to_string(index=False))
    print("-" * 60)
    print(f"Subtotal:          {receipt_df['Amount'].sum():.2f}")
    print(f"Discounted Total:  {receipt_df['Discounted (50%)'].sum():.2f}")
    print("=" * 60)
    # Recalculate inventory after deductions
    df['Amount'] = df['qty'] * df['prices']
    df['Discounted'] = df['Amount'] * 0.5
    return df
def main():
    df = load_inventory()
    while True:
        print("\n" + "=" * 40)
        print("       INVENTORY MANAGER")
        print("=" * 40)
        print("1. View Inventory")
        print("2. Add Product")
        print("3. Update Product")
        print("4. Remove Product")
        print("5. Low Stock Alert")
        print("6. Generate Receipt")
        print("7. Save & Exit")
        print("0. Exit without saving")
        choice = input("\nChoose an option: ").strip()
        if choice == '1':
            display_inventory(df)
        elif choice == '2':
            df = add_product(df)
        elif choice == '3':
            df = update_product(df)
        elif choice == '4':
            df = remove_product(df)
        elif choice == '5':
            threshold = input("Low stock threshold (default 5): ").strip()
            threshold = int(threshold) if threshold.isdigit() else 5
            low_stock_alert(df, threshold)
        elif choice == '6':
            df = generate_receipt(df)
        elif choice == '7':
            save_inventory(df)
            print("Goodbye!")
            break
        elif choice == '0':
            print("Exited without saving.")
            break
        else:
            print("Invalid option.")
if __name__ == "__main__":
    main()


       INVENTORY MANAGER
1. View Inventory
2. Add Product
3. Update Product
4. Remove Product
5. Low Stock Alert
6. Generate Receipt
7. Save & Exit
0. Exit without saving



Choose an option:  1



prod_name  qty  prices   Amount  Discounted
    Tecno   10 70000.0 700000.0    350000.0
  Infinix    5 90000.0 450000.0    225000.0
Total Stock Value (full price): 1150000.00
Total Stock Value (50% discount): 575000.00

       INVENTORY MANAGER
1. View Inventory
2. Add Product
3. Update Product
4. Remove Product
5. Low Stock Alert
6. Generate Receipt
7. Save & Exit
0. Exit without saving


# Day Seven

In [4]:
print("=== Safe number conversion ===")
user_input = "rei"          # try changing this to "hello"
try:
   number = int(user_input)
   print(f"Success! You entered the number {number}")
except ValueError:
   print("Error: That is not a whole number. Please try again.")
print("Program continues normally...")

=== Safe number conversion ===
Error: That is not a whole number. Please try again.
Program continues normally...


In [9]:
def safe_divide(a, b):
    try:
        result = a / b
        print(f"{a} ÷ {b} = {result}")
        return result
    except ZeroDivisionError:
        print("Error: You cannot divide by zero!")
        return None
    except TypeError:
        print("Error: Both values must be numbers.")
        return None
safe_divide(10,"zero")

Error: Both values must be numbers.
